## Activity Folding

**Modification Objective:** Combine multiple activity types into a single activity type when their distinction is not required.

**Motivation:** Event logs may represent events through separate activity types even though their distinction is unnecessary. Folding such activity types can provide a simplified list of activity types and reduce distinctions that are not relevant.

**Precondition:** The activity types of the events to be folded, the activity type through which they should be represented jointly, and the  rule for determining the new representation are specified.

**Approach:** Combine the selected activity types into a common activity type while optionally preserving the information required to represent the associated events according to the specified  rule.

**Output:** An event log in which the selected distsinct activity types are represented through a common activity type rather than as separate activities, with the distinction between them made explicit through attribute values.

**Implemented Example**: Sepsis log - for each case, rename activities named "Release" followed by a single letter (e.g., "Release A") to just "Release" and add a column "ReleaseCode" holding the letter that was part of the original activity name (missing for all other events)

In [ ]:
import pandas as pd
import pm4py

# --- Configuration -----------------------------------------------------------

LOG_PATH = "../../data/SepsisCases2020EventLog.xes"

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
COMPLETION_TIME = "time:timestamp"

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

display(event_log.head())

## Pattern execution

Example specific to sepsis event log, where all events that are of activity type Release... must be summarized for further analysis and a new attribute *ReleaseCode* is to be added. 

In [ ]:
# Fold "Release <letter>" activity variants into "Release", keeping the letter as ReleaseCode
modified_event_log = event_log.copy()


release_code = modified_event_log[ACTIVITY].str.extract(r'^Release ([A-Za-z])$')[0]

modified_event_log['ReleaseCode'] = release_code
modified_event_log.loc[release_code.notna(), ACTIVITY] = 'Release'

display(modified_event_log[modified_event_log['ReleaseCode'].notna()][[CASE_ID, ACTIVITY, 'ReleaseCode']])
display(modified_event_log[ACTIVITY].value_counts())

## Activity Unfolding

**Modification Objective:** Separate semantically distinct activities implicitly represented by only one  existing activity type.

**Motivation:** A recorded event may implicitly represent an activity that is semantically distinct from its assigned activity type. For example, an attribute value recorded at an event may result from a separate activity that is not represented by an activity type of its own. Keeping such information implicit can obscure the information represented by the event log. Making them explicit as distinct activity types can provide a representation that better reflects the occurrences evidenced by the recorded data.

**Precondition:** Semantically distinct activities have been identified as being implicitly represented by one existing activity type, and a distinguishing  rule (either at the case-level or event-level) is specified.

**Approach:** Change the activity type of implicitly represented activities by assigning them a distinct activity type based on the specified rule.

**Output:** An event log in which the previously implicit activity types are represented explicitly.

**Implemented Example**: RTFM Log - For each event named "Payment", rename it to "PartialPayment" if `paymentAmount` is less than `totalPaymentAmount`, otherwise rename it to "FullPayment"

In [ ]:
import pandas as pd
import pm4py

# --- Configuration -----------------------------------------------------------

LOG_PATH = "../../data/Road_Traffic_Fine_Management_Process.xes"

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
COMPLETION_TIME = "time:timestamp"

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

display(event_log.head())

## Pattern execution

Example specific to RTFM event log, where activities *PartialPayment* and *FullPayment* are to be added. 

In [ ]:
# Unfold "Payment" into "PartialPayment"/"FullPayment" based on paymentAmount vs totalPaymentAmount
modified_event_log = event_log.copy()

payment = modified_event_log[ACTIVITY] == 'Payment'
is_partial = modified_event_log['paymentAmount'] < modified_event_log['totalPaymentAmount']



modified_event_log.loc[payment & is_partial, ACTIVITY] = 'PartialPayment'
modified_event_log.loc[payment & ~is_partial, ACTIVITY] = 'FullPayment'

display(modified_event_log[modified_event_log[ACTIVITY].isin(['PartialPayment', 'FullPayment'])][[CASE_ID, ACTIVITY, 'paymentAmount', 'totalPaymentAmount']])
display(modified_event_log[ACTIVITY].value_counts())